# Pandas Exercises

Inspired by Tamás Gál

The latest version of this notebook is available at [https://github.com/escape2020/school2021](https://github.com/escape2020/school2021)

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as ml
import sys
plt = ml.pyplot
ml.rcParams['figure.figsize'] = (10.0, 5.0)

print(f"Python version: {sys.version}\n"
      f"Pandas version: {pd.__version__}\n"
      f"NumPy version: {np.__version__}\n"
      f"Matplotlib version: {ml.__version__}\n"
      f"seaborn version: {sns.__version__}")

Python version: 3.12.8 | packaged by conda-forge | (main, Dec  5 2024, 14:19:53) [Clang 18.1.8 ]
Pandas version: 2.2.3
NumPy version: 2.0.2
Matplotlib version: 3.10.0
seaborn version: 0.13.2


In [2]:
from IPython.core.magic import register_line_magic

@register_line_magic
def shorterr(line):
    """Show only the exception message if one is raised."""
    try:
        output = eval(line)
    except Exception as e:
        print("\x1b[31m\x1b[1m{e.__class__.__name__}: {e}\x1b[0m".format(e=e))
    else:
        return output
    
del shorterr

In [3]:
import warnings
warnings.filterwarnings('ignore')  # annoying UserWarnings from Jupyter/seaborn which are not fixed yet

## Exercise 1

Use the `pd.read_csv()` function to create a `DataFrame` from the dataset `data/neutrinos.csv`.

In [4]:
%shorterr neutrinos = pd.read_csv('data/neutrinos.csv')

SyntaxError: invalid syntax (<string>, line 1)


### Problems encountered

- the first few lines represent a plain header and need to be skipped
- comments are indicated with `$` at the beginning of the line
- the column separator is `:`
- the decimal delimiter is `,`
- the index column is the first one
- there is a footer to be excluded
- footer exclusion only works with the Python-engine

### Solution to exercise 1

In [ ]:
!head -n 15 data/neutrinos.csv

In [5]:
neutrinos = pd.read_csv('data/neutrinos.csv',
                        skiprows=5,
                        comment='$',
                        sep=':',
                        decimal=',',
                        index_col=0,
                        skipfooter=1,
                        engine='python')

In [7]:
neutrinos.head(10)

,azimuth,zenith,bjorkeny,energy,pos_x,pos_y,pos_z,proba_track,proba_cscd
0,2.349537,1.116004,0.048998,3.3664,52.740,28.831,401.186,0.824351,0.175649
1,5.575786,1.742838,0.280471,3.8900,48.369,29.865,417.282,0.818363,0.181637
2,4.656125,2.686909,0.119843,3.2335,71.722,121.449,363.077,0.828343,0.171657
3,0.520486,1.939326,0.061315,4.7840,-47.592,-84.466,350.687,0.842315,0.157685
4,2.856970,1.678897,0.061465,3.9833,-25.518,24.362,391.891,0.862275,0.137725
5,5.519597,2.219014,0.151957,4.6678,31.394,59.333,398.322,0.970060,0.029940
6,6.249158,1.330957,0.361199,4.9289,45.767,84.271,453.102,0.810379,0.189621
7,4.732427,2.259213,0.256519,4.4547,-15.944,-9.902,345.655,0.826347,0.173653
8,3.254083,1.312848,0.075715,2.5227,6.506,-61.922,422.897,0.810379,0.189621
9,0.546133,1.423536,0.152195,4.6579,54.179,52.946,340.457,0.882236,0.117764


### Check the dtypes to make sure everthing is parsed correctly (and is not an `object`-array)

In [ ]:
neutrinos.dtypes  # everything's ok now ;)

## Exercise 2

Create a histogram of the neutrino energies.

### Solution to exercise 2

In [ ]:
neutrinos.energy.hist(bins=100)
plt.xlabel('Neutrino energy [GeV]');
plt.ylabel('Count');
plt.show()

# alternative:

neutrinos.hist('energy', bins=100)

## Exercise 3

Use the `pd.read_csv()` function to create a `DataFrame` from the dataset `data/reco.csv`.

### Problems encountered

- need to define index column

### Solution to exercise 3

In [ ]:
reco = pd.read_csv('data/reco.csv', index_col=0)
reco.head()

## Exercise 4

Combine the `neutrinos` and `reco` `DataFrames`  into a single `DataFrame`

pd.concat()

### Problems encountered

- need to define the right axis
- identical column names should be avoided

### Solution to exercise 4

In [ ]:
data = pd.concat([neutrinos, reco.add_prefix('reco_')], axis="columns")

In [ ]:
data.head(3)

In [ ]:
data.columns

## Exercise 5

Make a scatter plot to visualise the zenith reconstruction quality.

`data = pd.concat([neutrinos reco.add_prefix('reco_')], axis="columns")`

### Problems encountered

- `DataFrame.plot()` is not suited to do scatter plots in earlier Pandas versions (inverts axis, sets weird limits etc.)

### Solution to Ex. 5

In [ ]:
data.plot(x='zenith', y='reco_zenith', style='.');

### Solution to exercise 5, using `plt.scatter()`

Sometimes it's better not to fight against `DataFrame.plot()`, just switch to Matplotlib ;)

In [ ]:
fig, ax = plt.subplots()
# s change the dots size
# alpha change the transparency
ax.scatter(data['zenith'], data['reco_zenith'], s=1, alpha=0.05);
ax.set_xlabel('True zenith');
ax.set_ylabel('Reconstructed zenith');

### Solution to exercise 5, using `plt.hist2d()`

In [ ]:
fig, ax = plt.subplots()
counts, xedges, yedges, im = ax.hist2d(data['zenith'], data['reco_zenith'], bins=50);
ax.set_xlabel('True zenith');
ax.set_ylabel('Reconstructed zenith');
fig.colorbar(im)

## Exercise 6

Create a histogram of the cascade probabilities (__`neutrinos`__ dataset: `proba_cscd` column) for the energy ranges 1-5 GeV, 5-10 GeV, 10-20 GeV and 20-100 GeV.

### Naive solution to exercise 6

In [ ]:
mask = (neutrinos.energy >= 1) & (neutrinos.energy < 5)
neutrinos[mask].proba_cscd.hist(histtype='step', label='[0-5) GeV')

mask = (neutrinos.energy >= 5) & (neutrinos.energy < 10)
neutrinos[mask].proba_cscd.hist(histtype='step', label='[5-10) GeV')

mask = (neutrinos.energy >= 10) & (neutrinos.energy < 20)
neutrinos[mask].proba_cscd.hist(histtype='step', label='[10-20) GeV')

mask = (neutrinos.energy >= 20) & (neutrinos.energy < 100)
neutrinos[mask].proba_cscd.hist(histtype='step', label='[20-100) GeV')

plt.legend()
plt.xlabel('proba cscd')

### More elegant solution
If we have a lot of bins, the naive solution can be difficult to do

In [ ]:
ebins = [0, 5, 10, 20, 100]
# directly find the bins of each event using pd.cut
bin_index = pd.cut(neutrinos.energy, ebins, labels=False).values
bin_index

In [ ]:
for i in set(bin_index):
    plt.hist(neutrinos.proba_cscd[bin_index==i], label=f'[{ebins[i]}-{ebins[i+1]}) GeV', bins=30, histtype='step')
plt.legend()
plt.xlabel('proba cscd')

#### Or even quicker:

In [ ]:
ebins = [1, 5, 10, 20, 100]
neutrinos['ebin'] = pd.cut(neutrinos.energy, ebins, labels=False)
neutrinos.hist('proba_cscd', by='ebin', bins=50);

## Exercise 7

Create a 2D histogram showing the distribution of the `x` and `y` values of the starting positions (`pos_x` and `pos_y`) of the neutrinos. This is basically a 2D plane of the starting positions.

### Solution to exercise 7

In [ ]:
fig, ax = plt.subplots()
counts, xedges, yedges, im = plt.hist2d(data.pos_x, data.pos_y, bins=100, cmap='viridis')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('2D Plane')
ax.axis('equal')
fig.colorbar(im);

## Exercise 8

Check out `seaborn` (`import seaborn as sns`) and recreate the 2D histogram from Exercies 7.

### Solution to exercise 8

In [ ]:
sns.displot(data, x="pos_x", y="pos_y", cbar=True);

In [ ]:
sns.jointplot(data=data, x="pos_x", y="pos_y", s=2, alpha=0.2)

## Exercise 9

Create two histograms of the `azimuth` and `zenith` distribution side by side, in one plot (two subplots).

Try `pandas` built-in matplotlib wrapper and also the raw matplotlib library.

In [ ]:
data.head(2)

### Solution to exercise 9

In [ ]:
data.hist(['azimuth', 'zenith'], bins=100, figsize=(10, 3));

#### Solution using matplotib

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))

for idx, column in enumerate(['azimuth', 'zenith']):
    data[column].hist(bins=100, ax=axes[idx])  # zenith=0 is coming from above
    axes[idx].set_xlabel(column + ' [rad]')
    axes[idx].set_ylabel('count')

## Exercise 10

Split the data into two groups: `upgoing` and `downgoing`, based on the `zenith` value (`zenith == 0` is coming from above).

Try out `sns.stripplot` to verify your "cut" on the data!

In [12]:
neutrinos

,azimuth,zenith,bjorkeny,energy,pos_x,pos_y,pos_z,proba_track,proba_cscd
0,2.349537,1.116004,0.048998,3.366400,52.740,28.831,401.186,0.824351,0.175649
1,5.575786,1.742838,0.280471,3.890000,48.369,29.865,417.282,0.818363,0.181637
2,4.656125,2.686909,0.119843,3.233500,71.722,121.449,363.077,0.828343,0.171657
3,0.520486,1.939326,0.061315,4.784000,-47.592,-84.466,350.687,0.842315,0.157685
4,2.856970,1.678897,0.061465,3.983300,-25.518,24.362,391.891,0.862275,0.137725
...,...,...,...,...,...,...,...,...,...
60973,6.167531,1.320392,0.093289,6.696500,-25.916,91.493,444.696,0.804391,0.195609
60974,4.933224,0.729290,0.335327,22.188999,-68.489,-78.661,389.067,0.886228,0.113772
60975,1.028631,1.091732,0.325266,14.085000,-66.319,56.084,378.345,0.824351,0.175649
60976,5.993753,1.713249,0.087113,6.198300,-47.727,66.036,372.069,0.976048,0.023952


In [15]:
mask = neutrinos['zenith'] <= 1
neutrinos[mask]

,azimuth,zenith,bjorkeny,energy,pos_x,pos_y,pos_z,proba_track,proba_cscd
28,1.220674,0.827076,0.150155,4.129100,-68.304,-36.132,426.951,0.820359,0.179641
34,5.518050,0.622615,0.212684,4.850600,-55.273,-62.485,434.217,0.908184,0.091816
43,4.317651,0.598771,0.111045,3.476600,-90.115,-40.805,413.269,0.882236,0.117764
50,3.933836,0.929799,0.211483,4.102400,42.392,37.691,386.523,0.856287,0.143713
53,1.681261,0.887835,0.120063,4.806800,0.401,-69.587,490.990,0.830339,0.169661
...,...,...,...,...,...,...,...,...,...
60829,2.565487,0.775123,0.247123,8.453100,80.577,25.622,431.977,0.902196,0.097804
60835,4.122594,0.760728,0.544408,18.448999,17.795,38.706,364.424,0.948104,0.051896
60954,5.039511,0.121932,0.115936,7.656300,-3.162,44.428,456.375,0.848303,0.151697
60974,4.933224,0.729290,0.335327,22.188999,-68.489,-78.661,389.067,0.886228,0.113772


In [17]:
~mask

0         True
1         True
2         True
3         True
4         True
         ...  
60973     True
60974    False
60975     True
60976     True
60977    False
Name: zenith, Length: 60978, dtype: bool

In [16]:
neutrinos[~mask]

,azimuth,zenith,bjorkeny,energy,pos_x,pos_y,pos_z,proba_track,proba_cscd
0,2.349537,1.116004,0.048998,3.3664,52.740,28.831,401.186,0.824351,0.175649
1,5.575786,1.742838,0.280471,3.8900,48.369,29.865,417.282,0.818363,0.181637
2,4.656125,2.686909,0.119843,3.2335,71.722,121.449,363.077,0.828343,0.171657
3,0.520486,1.939326,0.061315,4.7840,-47.592,-84.466,350.687,0.842315,0.157685
4,2.856970,1.678897,0.061465,3.9833,-25.518,24.362,391.891,0.862275,0.137725
...,...,...,...,...,...,...,...,...,...
60971,4.790261,3.022355,0.102240,7.3018,-22.131,-17.723,325.959,0.822355,0.177645
60972,4.552834,2.218202,0.017767,8.3767,-63.705,122.700,325.507,0.934132,0.065868
60973,6.167531,1.320392,0.093289,6.6965,-25.916,91.493,444.696,0.804391,0.195609
60975,1.028631,1.091732,0.325266,14.0850,-66.319,56.084,378.345,0.824351,0.175649


### Solution to exercise 10

Here, we are adding a new column to our dataset which contains True/False for each entry, regarding of its zenith direction

In [ ]:
data['upgoing'] = data.zenith < np.pi/2

In [ ]:
data_by_upgoing = data.groupby('upgoing')

Seaborn automatically recognises the grouped Pandas DataFrame:

In [ ]:
sns.stripplot(x="upgoing", y="zenith", data=data);

## Exercise 11

Create a combined histogram (two histograms overlayed in the same plot) for both `upgoing` and `downgoing` datasets, showing the `zenith` angle.

### Solution to exercise 11

In [ ]:
fig, ax = plt.subplots()

for upgoing, sub_data in data_by_upgoing:
    sub_data.hist('zenith', ax=ax, bins=100,
                  label='upgoing' if upgoing else 'downgoing',
                  histtype='step', linewidth=2)
ax.legend();